# Export Llama 3.2 1B Instruct to ExecuTorch (.pte) for the Jetson

Runs the same recipe as `scripts/export_llama_pte.sh` / `configs/llm/llama3_2_1b_instruct_8da4w.yaml`
(ExecuTorch **1.0.0**, XNNPACK, 8da4w, group 32, 4-bit embeddings, 2k context). The Jetson runners are built
from ExecuTorch v1.0.0, so the version here must stay 1.0.0.

1. Runtime: the default **CPU** runtime is enough (about 12.7 GB RAM). No GPU needed.
2. Run all cells. The login cell asks for a Hugging Face token (Read) for an account that has access to
   `meta-llama/Llama-3.2-1B-Instruct`.
3. The last cell downloads `llama3_2_1b_instruct_8da4w.pte` (~1 GB), `tokenizer.model` and `EXPORT_INFO.txt`.
   Put them in `~/Downloads` on the laptop.

Takes about 15-25 minutes.

In [ ]:
# fresh runtime: install before anything imports torch
!pip install -q "executorch==1.0.0" pytorch-tokenizers hydra-core omegaconf tiktoken sentencepiece blobfile huggingface_hub
!python -c "import executorch.version as v, torch, sys; print('python', sys.version.split()[0], 'executorch', v.__version__, 'torch', torch.__version__); print('' if v.__version__.startswith('1.0.0') else '!! executorch is not 1.0.0 (no 1.0.0 wheel for this Python?). The .pte still loaded on the 1.0.0 Jetson runtime in our test, but record the version in the report.')"
!free -g

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import snapshot_download
CKPT = "/content/Llama-3.2-1B-Instruct"
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"], local_dir=CKPT)
!ls -la {CKPT}/original

In [ ]:
RECIPE = r'''# ExecuTorch export recipe for the advisory model (executorch==1.0.0, extension.llm.export.export_llm).
# scripts/export_llama_pte.sh fills the @CKPT@ / @OUT@ placeholders and runs it.
base:
  model_class: llama3_2
  checkpoint: "@CKPT@/original/consolidated.00.pth"
  params: "@CKPT@/original/params.json"
  # BOS / EOS ids of Llama 3.x: generation stops at <|eot_id|> (128009) or <|end_of_text|> (128001)
  metadata: '{"get_bos_id":128000,"get_eos_ids":[128009,128001]}'
model:
  use_kv_cache: true
  use_sdpa_with_kv_cache: true    # fused custom SDPA op, much faster on CPU
  dtype_override: fp32            # recommended for XNNPACK
backend:
  xnnpack:
    enabled: true
    extended_ops: true
quantization:
  qmode: 8da4w                    # int8 dynamic activations x int4 weights for linear layers
  group_size: 32
  embedding_quantize: "4,32"      # 4-bit embedding table, group 32
export:
  max_seq_length: 2048            # longest prompt chunk prefilled at once
  max_context_length: 2048        # KV-cache window; advisory prompt ~0.7-1k tokens + <=200 generated
  output_name: "@OUT@"
'''

PTE = "/content/out_llm/llama3_2_1b_instruct_8da4w.pte"
!mkdir -p /content/out_llm
open("/content/export_config.yaml", "w").write(RECIPE.replace("@CKPT@", CKPT).replace("@OUT@", PTE))
print(open("/content/export_config.yaml").read())

In [ ]:
!/usr/bin/time -v python -m executorch.extension.llm.export.export_llm --config /content/export_config.yaml 2>&1 | tee /content/out_llm/export.log | grep -vE "^\s*$" | tail -40
!ls -la /content/out_llm

In [ ]:
import hashlib, subprocess, datetime, shutil
shutil.copy(f"{CKPT}/original/tokenizer.model", "/content/out_llm/tokenizer.model")
h = hashlib.sha256(open(PTE, "rb").read()).hexdigest()
ver = lambda m: subprocess.run(["python", "-c", f"import {m}; print({m}.__version__)"], capture_output=True, text=True).stdout.strip()
info = (f"model=meta-llama/Llama-3.2-1B-Instruct  executorch=={ver('executorch.version')}\n"
        f"torch=={ver('torch')}  torchao=={ver('torchao')}\n"
        "recipe=configs/llm/llama3_2_1b_instruct_8da4w.yaml (8da4w, group 32, 4-bit embeddings, XNNPACK, 2048 ctx)\n"
        f"exported={datetime.datetime.now().isoformat(timespec='seconds')} host=colab-x86_64 sha256={h}\n")
open("/content/out_llm/EXPORT_INFO.txt", "w").write(info)
print(info)

In [ ]:
from google.colab import files
for f in ("EXPORT_INFO.txt", "tokenizer.model", "llama3_2_1b_instruct_8da4w.pte"):
    files.download(f"/content/out_llm/{f}")
# If the 1 GB browser download stalls, copy to Drive instead:
# from google.colab import drive; drive.mount('/content/drive'); !cp /content/out_llm/* /content/drive/MyDrive/